# NB04A — S2 quarterly three-year extraction and ROI completeness


In [ ]:
# 00 — CONTROL: 3-year seasonal composite ending in nominal UAV year.
# This creates new data, does NOT alter V3 or V3_1 models.
# Legacy ROI.shp is used to define eligible satellite training area and audit.
# 4 calendar quarters x (10 S2 spectral bands + clear count) = 44 bands.
# For retrospective data modelling only: imagery in UAV year may postdate exact UAV flight.


## 01 — Input manifest and ROI


In [1]:
%pip -q install earthengine-api geemap geedim rasterio geopandas shapely scikit-learn pandas matplotlib scipy tqdm
import json,os,math,hashlib,shutil,time,warnings
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from rasterio.warp import reproject,Resampling,transform_bounds
from affine import Affine
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():drive.mount("/content/drive")
ROOT=next((p for p in [Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final"),
                         Path("/content/drive/MyDrive/Musanga/Final")] if p.is_dir()),None)
assert ROOT is not None
LEGACY=ROOT.parent/"KVO_work/KVO_08_satellite_fraction"
LEGACY_GRIDS=LEGACY/"validation/09_s2_canonical_10m_grids.json"
MANIFEST=ROOT/"Outputs/V3/00_input_audit/00_input_manifest.json"
assert MANIFEST.exists(),MANIFEST
A=json.loads(MANIFEST.read_text())
assert A.get("schema_version")=="v3_input_audit_5_nadine_circles"
# NB01 scientific contract: five mosaics, four forest groups.
# MOS 3–4 was deliberately excluded from model development/validation.
EXCLUDED_MOSAIC="Mos_3_4_transparent_mosaic_group1_georef.tif"
assert EXCLUDED_MOSAIC in [m["file"] for m in A["mosaics"]]
SOURCES={m["file"]:dict(site=m["site"],path=Path(m["path"]))
         for m in A["mosaics"] if m["file"]!=EXCLUDED_MOSAIC}
assert len(SOURCES)==5 and all(q["path"].exists() for q in SOURCES.values())
assert set(q["site"] for q in SOURCES.values())=={"Baego","MOS","Yangambi","Yoko"}
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union
ROI_FILE=Path(A["roi"])
assert ROI_FILE.name.casefold()=="roi.shp" and ROI_FILE.exists(),ROI_FILE
ROI_GDF=gpd.read_file(ROI_FILE)
assert ROI_GDF.crs is not None and len(ROI_GDF)>0
assert ROI_GDF.geometry.notna().all() and ROI_GDF.geometry.is_valid.all(),(
    "ROI.shp invalid: use validated V3 NB01 geometries")
ROI_SHA=hashlib.sha256(b"".join(
    (ROI_FILE.with_suffix(ext).read_bytes() if ROI_FILE.with_suffix(ext).exists() else b"")
    for ext in (".shp",".shx",".dbf",".prj"))).hexdigest()
assert ROI_SHA
print("EXCLUDED model mosaic:",EXCLUDED_MOSAIC)
print("Modelling mask:",ROI_FILE,"geometries:",len(ROI_GDF),"SHA:",ROI_SHA[:16])
YEARS={"Baego":2023,"Yoko":2020,"Yangambi":2024,"MOS":2024,"Luki":2023}
PROVENANCE={"Baego":"V2/08","Yoko":"V2/08","Yangambi":"V2/08",
            "MOS":"V2/08 for MOS_1_2; extrapolated provisionally to MOS_3_4/MOS_5_6",
            "Luki":"V2/08; exact acquisition date unresolved"}
INCLUDE_LUKI=False
if INCLUDE_LUKI:
    p=ROOT.parent/"data/Geo_Ref/Luki.tif"
    assert p.exists()
    SOURCES["Luki.tif"]=dict(site="Luki",path=p)
SITES=sorted(set(x["site"] for x in SOURCES.values()))
OUT=ROOT/"Outputs/V3_1/04A_S2_seasonal_3yr"
GRID_DIR=OUT/"grid";FEAT_DIR=OUT/"features";# Preserve previous unmasked aggregation files; new outputs are ROI-specific.
LEGACY_AGG_DIR=OUT/"uav_aggregated"
AGG_DIR=OUT/"uav_aggregated_ROI"
MODEL_DIR=OUT/"models_ROI";QA_DIR=OUT/"qa_ROI"
for p in (GRID_DIR,FEAT_DIR,AGG_DIR,MODEL_DIR,QA_DIR):p.mkdir(parents=True,exist_ok=True)
MODE="final_all30"   # NB03 model, must match native RGB probability maps
NB03=ROOT/("Outputs/V3/03_M2_full_mosaic_"+MODE)/"probability_maps"
THRESHOLD=0.5
MIN_UAV_COVERAGE=.90
MIN_FEATURE_VALID=.90
SCALES=[10,20,30,50,75,100]
assert set(SCALES)=={10,20,30,50,75,100}
assert all(x["site"] in YEARS for x in SOURCES.values())
reg=[]
for m,x in SOURCES.items():
    with rasterio.open(x["path"]) as d:
        assert d.crs is not None
        reg.append(dict(mosaic=m,site=x["site"],year=YEARS[x["site"]],
                        crs=str(d.crs),width=d.width,height=d.height,
                        raster_GB=round(x["path"].stat().st_size/1e9,3),
                        year_basis=PROVENANCE[x["site"]]))
display(pd.DataFrame(reg))
(OUT/"04_input_registry.json").write_text(json.dumps(
    dict(years=YEARS,provenance=PROVENANCE,sources={k:dict(site=v["site"],path=str(v["path"]))
    for k,v in SOURCES.items()},uav_model=MODE,roi=str(ROI_FILE),roi_sha=ROI_SHA,
    excluded_mosaic=EXCLUDED_MOSAIC),indent=2))
print("NB03 files may still be running. Satellite download blocks do not require NB03.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.2 MB/s eta 0:00:00
Mounted at /content/drive
EXCLUDED model mosaic: Mos_3_4_transparent_mosaic_group1_georef.tif
Modelling mask: /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Data/Ortho_RGB/ROI.shp geometries: 9 SHA: 4f253e1199245a98


,mosaic,site,year,crs,width,height,raster_GB,year_basis
0,Baego.tif,Baego,2023,EPSG:32635,89967,61291,1.193,V2/08
1,Mos_1_2_transparent_mosaic_group1_georef.tif,MOS,2024,EPSG:32635,63924,22137,5.661,V2/08 for MOS_1_2; extrapolated provisionally ...
2,Mos_5_6_transparent_mosaic_group1_georef.tif,MOS,2024,EPSG:32635,31639,39755,5.032,V2/08 for MOS_1_2; extrapolated provisionally ...
3,Yangambi_georef.tif,Yangambi,2024,EPSG:32635,101817,37433,5.428,V2/08
4,Yoko_kvo_georef.tif,Yoko,2020,EPSG:32635,44359,23217,2.198,V2/08


NB03 files may still be running. Satellite download blocks do not require NB03.


## 02 — Native S2 grids and ROI mask


In [2]:
assert LEGACY_GRIDS.exists(),(
    "V2/08 native Sentinel-2 grid registry missing; build it in V2/08 Block 02 first",LEGACY_GRIDS)
old=json.loads(LEGACY_GRIDS.read_text())
SITE_V2={"Baego":"Baego","Yangambi":"Yangambi","Yoko":"Yoko",
         "MOS":"MOS_1_2","Luki":"Luki"}
GRIDS={}
# Keep the existing exact native-S2 grid for cached satellite products.
# The grid may include pixels outside ROI; all training/evaluation/output maps
# are filtered by independently rasterizing ROI.shp.
def clipped_roi_geometry(site):
    paths=[x["path"] for x in SOURCES.values() if x["site"]==site]
    assert paths,site
    with rasterio.open(paths[0]) as ref:
        crs=ref.crs
    roi=ROI_GDF.to_crs(crs)
    footprints=[]
    for p in paths:
        with rasterio.open(p) as ds:
            assert ds.crs==crs
            footprints.append(box(*ds.bounds))
    geometry=unary_union(list(roi.geometry)).intersection(unary_union(footprints))
    assert not geometry.is_empty and geometry.area>0,(
        "ROI.shp does not intersect retained site mosaics",site)
    return geometry

ROI_BY_SITE={site:clipped_roi_geometry(site) for site in SITES}
def roi_fraction_grid(site,grid,subpixel=4):
    """Fraction of each satellite-scale pixel falling inside eligible ROI.shp."""
    from rasterio.features import rasterize
    import rasterio
    from shapely.geometry import mapping
    crs=grid["crs"]
    with rasterio.open(next(x["path"] for x in SOURCES.values() if x["site"]==site)) as ds:
        geo_crs=ds.crs
    assert str(geo_crs)==crs
    h,w=grid["height"],grid["width"]
    affine=Affine(*grid["transform"])
    # Rasterize only the actual eligible site ROI, not the full image bbox.
    fine=rasterize([(ROI_BY_SITE[site],1)],out_shape=(h*subpixel,w*subpixel),
        transform=affine*Affine.scale(1/subpixel,1/subpixel),
        fill=0,dtype="uint8",all_touched=False)
    return fine.reshape(h,subpixel,w,subpixel).mean(axis=(1,3)).astype("float32")
ROI_MIN_FRACTION=0.90
assert 0<ROI_MIN_FRACTION<=1
for site in SITES:
    key=SITE_V2[site]
    assert key in old,("Missing V2 grid",site,key)
    g=old[key]
    assert int(g["uav_year"])==YEARS[site],(site,g["uav_year"])
    crs=g["uav_crs"]
    # ORIGINAL native scene grid: universal origin for *all* output scales,
    # independent of each site's mosaic footprint.
    native=Affine(*g["native_scene_transform"])
    assert abs(native.a-10)<1e-6 and abs(native.e+10)<1e-6
    assert abs(native.b)<1e-9 and abs(native.d)<1e-9
    rasters=[q["path"] for q in SOURCES.values() if q["site"]==site]
    bounds=[]
    for p in rasters:
        with rasterio.open(p) as ds:
            assert str(ds.crs)==crs,("Mosaic and canonical S2 CRS differ",site,p,ds.crs,crs)
            bounds.append(ds.bounds)
    west=min(b.left for b in bounds);south=min(b.bottom for b in bounds)
    east=max(b.right for b in bounds);north=max(b.top for b in bounds)
    scale=10
    x0=native.c+math.floor((west-native.c)/scale)*scale
    y0=native.f-math.floor((native.f-north)/scale)*scale
    nx=math.ceil((east-x0)/scale);ny=math.ceil((y0-south)/scale)
    GRIDS[site]=dict(site=site,year=YEARS[site],crs=crs,
        native_scene_transform=list(native)[:6],transform=[10,0,x0,0,-10,y0],
        width=nx,height=ny,bounds=[west,south,east,north],
        mosaics=[k for k,q in SOURCES.items() if q["site"]==site],
        scale=10,s2_native_tile=g["s2_native_tile"])
    print(site,"S2 native CRS",crs,"grid",nx,ny,"year",YEARS[site],
          "tile",g["s2_native_tile"])
(OUT/"04_canonical_site_grids.json").write_text(json.dumps(GRIDS,indent=2))

def scaled_grid(site,scale):
    """Exact nested physical UTM grid; does not assume scale divides 10."""
    g=GRIDS[site];native=Affine(*g["native_scene_transform"])
    west,south,east,north=g["bounds"]
    x0=native.c+math.floor((west-native.c)/scale)*scale
    y0=native.f-math.floor((native.f-north)/scale)*scale
    nx=math.ceil((east-x0)/scale);ny=math.ceil((y0-south)/scale)
    return dict(site=site,scale_m=scale,crs=g["crs"],
                transform=[scale,0,x0,0,-scale,y0],width=nx,height=ny)
for scale in SCALES:
    assert all(scaled_grid(s,scale)["width"]>0 for s in SITES)
print("Canonical grids ready. Features can now be downloaded independently of NB03.")


Baego S2 native CRS EPSG:32635 grid 721 491 year 2023 tile 35NLA
MOS S2 native CRS EPSG:32635 grid 784 913 year 2024 tile 35NLA
Yangambi S2 native CRS EPSG:32635 grid 509 187 year 2024 tile 35NKA
Yoko S2 native CRS EPSG:32635 grid 320 166 year 2020 tile 35NKA
Canonical grids ready. Features can now be downloaded independently of NB03.


## 03 — Three-year quarterly export (year−2 to UAV year)


In [10]:
import ee,geemap,geedim
EE_PROJECT="musanga-511110"
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
S2_BANDS=["B2","B3","B4","B8","B5","B6","B7","B8A","B11","B12"]
S2_INDEX=["NDVI","NDMI","NBR","NDRE"]
EMB_BANDS=[f"A{k:02d}" for k in range(64)]
FAMILIES={
 "RAW_ANNUAL":S2_BANDS+S2_INDEX+["N_CLEAR_S2"],
 "S2_QUARTERLY":[f"Q{q}_{band}" for q in range(1,5)
                 for band in S2_BANDS+["N_CLEAR"]],
 "EMB_ANNUAL":EMB_BANDS
}
S2_SOURCE="COPERNICUS/S2_SR_HARMONIZED"
EMB_SOURCE="GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
SCL_REJECT=[0,1,3,7,8,9,10,11]
def region(site):
    g=GRIDS[site]
    from rasterio.warp import transform_bounds
    aff=Affine(*g["transform"])
    west=aff.c;south=aff.f-g["height"]*10
    east=aff.c+g["width"]*10;north=aff.f
    b=transform_bounds(g["crs"],"EPSG:4326",west,south,east,north,densify_pts=21)
    return ee.Geometry.Rectangle(list(b),proj="EPSG:4326",geodesic=False)
def clean_s2(im):
    scl=im.select("SCL")
    valid=ee.Image(1)
    for x in SCL_REJECT:valid=valid.And(scl.neq(x))
    valid=valid.And(im.select("B8A").mask()).And(im.select("B9").mask())
    # Preserve native reflectances until post-composite 0.0001 scaling.
    out=ee.Image.cat([im.select(S2_BANDS[:4]),
                      im.select(S2_BANDS[4:]).resample("bilinear")])
    return out.updateMask(valid).copyProperties(im,["system:time_start"])
def add_indices(im):
    return ee.Image.cat([im,
        im.normalizedDifference(["B8","B4"]).rename("NDVI"),
        im.normalizedDifference(["B8","B11"]).rename("NDMI"),
        im.normalizedDifference(["B8","B12"]).rename("NBR"),
        im.normalizedDifference(["B8A","B5"]).rename("NDRE")])
def make_satellite_image(site,family):
    yr=YEARS[site]
    area=region(site)
    if family=="EMB_ANNUAL":
        col=(ee.ImageCollection(EMB_SOURCE)
            .filterDate(f"{yr}-01-01",f"{yr+1}-01-01").filterBounds(area))
        n=int(col.size().getInfo())
        assert n>0,(site,yr,"Annual embedding unavailable")
        return col.mosaic().select(EMB_BANDS).toFloat().clip(area)
    s2=(ee.ImageCollection(S2_SOURCE)
        .filterDate(f"{yr-2}-01-01",f"{yr+1}-01-01")
        .filterBounds(area).map(clean_s2))
    assert int(s2.size().getInfo())>0,(site,yr,"No S2 SR scenes")
    if family=="RAW_ANNUAL":
        med=s2.median().multiply(.0001)
        count=s2.select("B2").count().unmask(0).toFloat().rename("N_CLEAR_S2")
        return ee.Image.cat([add_indices(med),count]).toFloat().clip(area)
    if family=="S2_QUARTERLY":
        layers=[]
        for quarter in (1,2,3,4):
            first=f"{yr}-{quarter*3-2:02d}-01"
            last=f"{yr+1}-01-01" if quarter==4 else f"{yr}-{quarter*3+1:02d}-01"
            col=s2.filter(ee.Filter.calendarRange(quarter*3-2,quarter*3,"month"))
            assert int(col.size().getInfo())>0,(site,yr,quarter,"No seasonal scenes")
            med=col.median().multiply(.0001)
            layers.append(med.rename([f"Q{quarter}_{b}" for b in S2_BANDS]))
            layers.append(col.select("B2").count().unmask(0).toFloat().rename(f"Q{quarter}_N_CLEAR"))
        return ee.Image.cat(layers).toFloat().clip(area)
    raise ValueError(family)

# FIRST RUN: one small controlled feature batch. Expand after success.
DOWNLOAD_SITES= ["Baego", "MOS", "Yangambi", "Yoko"]             # expand to SITES for full study
DOWNLOAD_FAMILIES=["S2_QUARTERLY"]  # three-year calendar-quarter medians
START_DOWNLOAD=True                    # execute to download; requires Colab+EE auth
assert set(DOWNLOAD_SITES).issubset(SITES)
assert set(DOWNLOAD_FAMILIES).issubset(FAMILIES)
LOCAL_DOWNLOAD=Path("/content/nb04_satellite_download")
LOCAL_DOWNLOAD.mkdir(exist_ok=True)
def feature_path(site,family):
    return FEAT_DIR/family/f"{site}_{family}_{YEARS[site]}_10m.tif"
def feature_ok(path,site,family):
    if not path.exists():return False
    g=GRIDS[site]
    try:
        with rasterio.open(path) as ds:
            return (ds.count==len(FAMILIES[family]) and ds.width==g["width"] and
                ds.height==g["height"] and str(ds.crs)==g["crs"] and
                ds.transform.almost_equals(Affine(*g["transform"]),precision=7) and
                ds.nodata==-9999 and ds.dtypes[0]=="float32" and
                ds.tags().get("FEATURE_FAMILY")==family and
                ds.tags().get("ACQUISITION_YEAR")==str(YEARS[site]))
    except Exception:return False
def download_feature(site,family):
    target=feature_path(site,family)
    target.parent.mkdir(exist_ok=True)
    if target.exists():
        assert feature_ok(target,site,family),("Incompatible old feature; inspect",target)
        print("CACHE HIT",site,family,target,flush=True)
        return
    g=GRIDS[site]
    tmp=LOCAL_DOWNLOAD/f"{site}_{family}_{YEARS[site]}.partial.tif"
    pending=target.with_name(target.name+".uploading")
    assert not pending.exists(),("Interrupted Drive upload; inspect",pending)
    if not tmp.exists():
        image=make_satellite_image(site,family).unmask(-9999,sameFootprint=False).toFloat()
        print("DOWNLOADING",site,family,len(FAMILIES[family]),"bands; grid",
              g["width"],"x",g["height"],flush=True)
        geemap.download_ee_image(
            image,filename=str(tmp),crs=g["crs"],crs_transform=g["transform"],
            shape=(g["height"],g["width"]),dtype="float32",
            resampling="near",max_tile_dim=256,num_threads=1,overwrite=False)
    # Validate *full raster content* and georeferencing before publication.
    with rasterio.open(tmp,"r+") as ds:
        assert ds.count==len(FAMILIES[family]),(site,family,ds.count)
        assert (ds.width,ds.height)==(g["width"],g["height"])
        assert str(ds.crs)==g["crs"] and ds.transform.almost_equals(
            Affine(*g["transform"]),precision=7)
        assert ds.dtypes[0]=="float32"
        for band in range(1,ds.count+1):
            for _,window in ds.block_windows(band):
                val=ds.read(band,window=window)
                assert np.isfinite(val).all(),("NaN/Inf in EE image",site,family,band)
            ds.set_band_description(band,FAMILIES[family][band-1])
        assert ds.nodata in (None,-9999) or (
            ds.nodata is not None and np.isneginf(ds.nodata)),(
            "Unexpected EE NoData",ds.nodata)
        ds.nodata=-9999.
        ds.update_tags(FEATURE_FAMILY=family,ACQUISITION_YEAR=str(YEARS[site]),
                       SOURCE_COLLECTION=S2_SOURCE if family!="EMB_ANNUAL" else EMB_SOURCE,
                       UTM_GRID_SOURCE="V2_08_native_S2_B2",BRDF="NONE",
                       FEATURE_BANDS=json.dumps(FAMILIES[family]))
    assert feature_ok(tmp,site,family)
    shutil.copyfile(tmp,pending)
    assert feature_ok(pending,site,family)
    os.replace(pending,target)
    tmp.unlink(missing_ok=True)
    print("SAVED",site,family,target,flush=True)
if START_DOWNLOAD:
    tasks=[(s,f) for s in DOWNLOAD_SITES for f in DOWNLOAD_FAMILIES]
    for j,(site,family) in enumerate(tasks,1):
        print(f"[{j}/{len(tasks)}] {site} {family}",flush=True)
        download_feature(site,family)
print("QUARTERLY 3-YEAR FILES, NOT EXACT UAV-DATE MATCHED")
print("Available verified satellite rasters:")
display(pd.DataFrame([dict(site=s,family=f,available=feature_ok(feature_path(s,f),s,f))
                      for s in SITES for f in FAMILIES]))


[1/8] Baego RAW_ANNUAL
CACHE HIT Baego RAW_ANNUAL /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3/04_satellite_Musanga/features/RAW_ANNUAL/Baego_RAW_ANNUAL_2023_10m.tif
[2/8] Baego EMB_ANNUAL
CACHE HIT Baego EMB_ANNUAL /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3/04_satellite_Musanga/features/EMB_ANNUAL/Baego_EMB_ANNUAL_2023_10m.tif
[3/8] MOS RAW_ANNUAL
DOWNLOADING MOS RAW_ANNUAL 15 bands; grid 784 x 913


/usr/local/lib/python3.13/dist-packages/geemap/common.py:12333: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)
/usr/local/lib/python3.13/dist-packages/geemap/common.py:12334: FutureWarning: 'num_threads' is deprecated and has no effect.  'max_requests' and 'max_cpus' can be used to limit concurrency.
  img.download(filename, overwrite=overwrite, num_threads=num_threads, **kwargs)


  0%|          |0/240 tiles [00:00<?]

/usr/local/lib/python3.13/dist-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


SAVED MOS RAW_ANNUAL /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3/04_satellite_Musanga/features/RAW_ANNUAL/MOS_RAW_ANNUAL_2024_10m.tif
[4/8] MOS EMB_ANNUAL
DOWNLOADING MOS EMB_ANNUAL 64 bands; grid 784 x 913


...

  0%|          |0/2 tiles [00:00<?]

SAVED Yoko RAW_ANNUAL /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3/04_satellite_Musanga/features/RAW_ANNUAL/Yoko_RAW_ANNUAL_2020_10m.tif
[8/8] Yoko EMB_ANNUAL
DOWNLOADING Yoko EMB_ANNUAL 64 bands; grid 320 x 166


  0%|          |0/8 tiles [00:00<?]

SAVED Yoko EMB_ANNUAL /content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3/04_satellite_Musanga/features/EMB_ANNUAL/Yoko_EMB_ANNUAL_2020_10m.tif
Available verified satellite rasters:


,site,family,available
0,Baego,RAW_ANNUAL,True
1,Baego,S2_QUARTERLY,False
2,Baego,EMB_ANNUAL,True
3,MOS,RAW_ANNUAL,True
4,MOS,S2_QUARTERLY,False
5,MOS,EMB_ANNUAL,True
6,Yangambi,RAW_ANNUAL,True
7,Yangambi,S2_QUARTERLY,False
8,Yangambi,EMB_ANNUAL,True
9,Yoko,RAW_ANNUAL,True


## 04 — Coverage QA


In [ ]:
# ROI-based quarterly completeness audit. Run AFTER download.
from rasterio.features import rasterize
rows=[]
for site in DOWNLOAD_SITES:
    p=feature_path(site,"S2_QUARTERLY")
    info=dict(site=site,start_year=YEARS[site]-2,end_year=YEARS[site],file=str(p),exists=p.exists())
    if p.exists():
        with rasterio.open(p) as d:
            mask=roi_fraction_grid(site,GRIDS[site])>=ROI_MIN_FRACTION
            info["roi_cells"]=int(mask.sum())
            assert mask.any(),(site,"ROI does not intersect raster")
            for q in range(1,5):
                clear=d.read(q*11)
                valid=np.isfinite(clear)&(clear>=1)
                info[f"Q{q}_clear_ge1_frac"]=float(valid[mask].mean())
                info[f"Q{q}_mean_clear_count"]=float(np.maximum(clear[mask],0).mean())
            info["complete_90pct_each_quarter"]=all(info[f"Q{q}_clear_ge1_frac"]>=.90 for q in range(1,5))
    rows.append(info)
audit=pd.DataFrame(rows)
display(audit)
audit.to_csv(OUT/"quarterly_3yr_ROI_completeness.csv",index=False)
print("WARNING: UAV years only; exact acquisition dates not verified.")
